# 분류기 학습 결과 분석

`train_classifier.ipynb`로 학습이 끝난 실행 폴더의 결과 파일만 읽어 그림과 표를 만든다. 학습이나 평가를 다시 하지 않는다.

- 읽는 파일: `<run_dir>/metrics.csv`, `val_predictions.json`, `class_map.json`, `data_summary.json`, 데이터 폴더의 `samples.csv`, `crops/`
- 그림 저장: `<run_dir>/analysis/*.png` (보고서용)
- 설정 셀의 `run_name`만 학습 때와 같게 맞추면 된다. Colab은 학습 노트북과 같은 경로(Drive 결과 폴더, `/content/training_package_classifier`)를 쓴다.

| 절 | 내용 | 계획서 |
|---|---|---|
| 1 | 학습 곡선과 best epoch | 7절 7번 |
| 2 | 전체 val과 유사 사진 제외 val 점수 | 9절 (추가: `near_dup_in_train`) |
| 3 | 클래스별 recall·F1, train 샘플 수와의 관계 | 9절, 7절 8번 |
| 4 | 자주 혼동되는 품목 쌍과 예시 사진 | 9절 |
| 5 | 대표 오분류 사진 | 7절 8번 |

In [ ]:
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.metrics import f1_score

IN_COLAB = "google.colab" in sys.modules
project_root = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file()
)

run_name = "resnet50_baseline_v1"  # 학습 때의 run_name

if IN_COLAB:
    run_dir = Path("/content/drive/MyDrive/gitkk_runs/classifier") / run_name
    data_dir = Path("/content/training_package_classifier")
else:
    run_dir = Path(os.environ.get("CLASSIFIER_RUN_DIR", project_root / "outputs" / run_name))
    data_dir = Path(os.environ.get("CLASSIFIER_DATA_DIR", project_root / "data" / "training_package_classifier"))
fig_dir = run_dir / "analysis"
fig_dir.mkdir(exist_ok=True)

metrics = pd.read_csv(run_dir / "metrics.csv")
predictions = pd.DataFrame(json.loads((run_dir / "val_predictions.json").read_text(encoding="utf-8")))
class_map = json.loads((run_dir / "class_map.json").read_text(encoding="utf-8"))
data_summary = json.loads((run_dir / "data_summary.json").read_text(encoding="utf-8"))
samples = pd.read_csv(data_dir / "samples.csv", dtype={"item_seq": str})

classes = pd.DataFrame(class_map["classes"]).set_index("class_index")
classes["code"] = classes.category_id.map(lambda c: f"K-{c:06d}")  # 그림에는 한글 대신 약 코드
classes["train_count"] = [data_summary["train_class_counts"][str(i)] for i in classes.index]
predictions = predictions.merge(samples[["sample_id", "crop_path", "near_dup_in_train"]], on="sample_id", how="left")
assert predictions.crop_path.notna().all(), "samples.csv가 이 실행의 데이터와 다릅니다"

print(f"run_dir: {run_dir}")
print(f"data_dir: {data_dir}")
print(f"epoch 수: {len(metrics)}, val 예측: {len(predictions)}개, 오분류: {(~predictions.correct).sum()}개")

## 1. 학습 곡선과 best epoch

best epoch는 계획서 기준(val macro F1 최대, 같으면 Top-1이 높은 epoch)으로 표시한다. 회색 영역은 head-only 단계다.

In [ ]:
m = metrics.copy()
m["epoch"] += 1  # 1부터 표시
best = m.sort_values(["val_macro_f1", "val_top1_accuracy"], ascending=False).iloc[0]
head_only_end = m.loc[m.training_stage == "head_only", "epoch"].max()

panels = [
    ("Loss", ["train_loss", "val_loss"]),
    ("Top-1 accuracy", ["train_top1_accuracy", "val_top1_accuracy"]),
    ("Top-5 accuracy", ["train_top5_accuracy", "val_top5_accuracy"]),
    ("Val macro F1", ["val_macro_f1"]),
]
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, (title, cols) in zip(axes.flat, panels):
    for col in cols:
        ax.plot(m.epoch, m[col], marker="o", markersize=3, label=col)
    if pd.notna(head_only_end):
        ax.axvspan(0.5, head_only_end + 0.5, color="0.9", zorder=0)
    ax.axvline(best.epoch, color="red", linestyle="--", linewidth=1, label=f"best epoch {int(best.epoch)}")
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(fig_dir / "learning_curves.png", dpi=150)
plt.show()

print(f"best epoch: {int(best.epoch)} ({best.training_stage})")
print(best[["val_loss", "val_top1_accuracy", "val_top5_accuracy", "val_macro_f1", "validation_class_coverage"]].to_string())

## 2. 전체 val과 유사 사진 제외 val

macro F1은 계획서 9절대로 그 val에 한 장 이상 있는 클래스만 대상으로 계산한다. `near_dup_in_train=True`는 train에 거의 같은 사진(알약 하나만 다른 조합)이 있는 val 샘플이다.

In [ ]:
def summarize(d):
    labels = sorted(d.target_class_index.unique())
    return pd.Series({
        "샘플 수": len(d),
        "Top-1": d.correct.mean(),
        "macro F1": f1_score(d.target_class_index, d.predicted_class_index, labels=labels, average="macro", zero_division=0),
        "클래스 수": len(labels),
    })

subsets = {
    "전체 val": predictions,
    "유사 사진 제외 val": predictions[~predictions.near_dup_in_train],
    "유사 사진만": predictions[predictions.near_dup_in_train],
}
score_table = pd.DataFrame({name: summarize(d) for name, d in subsets.items()}).T
score_table.to_csv(fig_dir / "val_subset_scores.csv", encoding="utf-8")
score_table.style.format({"샘플 수": "{:.0f}", "Top-1": "{:.4f}", "macro F1": "{:.4f}", "클래스 수": "{:.0f}"})

## 3. 클래스별 성능

`best.pt`의 val 예측으로 클래스마다 precision, recall, F1을 계산한다. train 샘플이 적은 클래스의 recall이 낮은지 함께 본다.

In [ ]:
per_class = []
for idx, row in classes.iterrows():
    t = predictions.target_class_index == idx
    p = predictions.predicted_class_index == idx
    tp = (t & p).sum()
    support = int(t.sum())
    precision = tp / p.sum() if p.sum() else np.nan
    recall = tp / support if support else np.nan
    f1 = 2 * precision * recall / (precision + recall) if support and precision + recall > 0 else (0.0 if support else np.nan)
    per_class.append({"class_index": idx, "code": row.code, "name": row["name"], "train_count": row.train_count,
                      "support": support, "precision": precision, "recall": recall, "f1": f1})
per_class = pd.DataFrame(per_class)
per_class.to_csv(fig_dir / "per_class_metrics.csv", index=False, encoding="utf-8")

worst = per_class.dropna(subset=["recall"]).sort_values(["recall", "f1", "support"]).head(20)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
ax = axes[0]
ax.barh(worst.code[::-1], worst.recall[::-1])
for y, (r, s) in enumerate(zip(worst.recall[::-1], worst.support[::-1])):
    ax.text(r + 0.005, y, f"{r:.2f} (n={s})", va="center", fontsize=8)
ax.set_xlim(0, 1.1)
ax.set_title("Lowest recall 20 classes (n = val support)")
ax.set_xlabel("Recall")

ax = axes[1]
ax.scatter(per_class.train_count, per_class.recall, s=per_class.support * 2, alpha=0.6)
for r in worst.head(5).itertuples():
    ax.annotate(r.code, (r.train_count, r.recall), fontsize=8)
ax.set_xlabel("Train samples per class")
ax.set_ylabel("Val recall")
ax.set_title("Recall vs train samples (point size = val support)")
fig.tight_layout()
fig.savefig(fig_dir / "per_class_recall.png", dpi=150)
plt.show()

worst[["code", "name", "train_count", "support", "precision", "recall", "f1"]].round(3)

## 4. 자주 혼동되는 품목 쌍

정답 → 예측 방향별로 센다. 아래 그림은 쌍마다 틀린 val crop 하나와, 예측한 클래스의 정답 val crop 하나(있으면)를 나란히 보여 준다.

In [ ]:
errors = predictions[~predictions.correct]
pairs = (errors.groupby(["target_class_index", "predicted_class_index"]).size()
         .rename("count").reset_index().sort_values("count", ascending=False))
pairs["target"] = pairs.target_class_index.map(classes.code)
pairs["predicted"] = pairs.predicted_class_index.map(classes.code)
pairs["target_name"] = pairs.target_class_index.map(classes["name"])
pairs["predicted_name"] = pairs.predicted_class_index.map(classes["name"])
pairs.to_csv(fig_dir / "confusion_pairs.csv", index=False, encoding="utf-8")


def load_crop(crop_path):
    return Image.open(data_dir / crop_path).convert("RGB")


top_pairs = pairs.head(8)
fig, axes = plt.subplots(len(top_pairs), 2, figsize=(5, 2.4 * len(top_pairs)), squeeze=False)
for row_axes, pr in zip(axes, top_pairs.itertuples()):
    wrong = errors[(errors.target_class_index == pr.target_class_index)
                   & (errors.predicted_class_index == pr.predicted_class_index)].iloc[0]
    ref = predictions[(predictions.target_class_index == pr.predicted_class_index) & predictions.correct]
    row_axes[0].imshow(load_crop(wrong.crop_path))
    row_axes[0].set_title(f"true {pr.target} -> pred {pr.predicted}\n({pr.count}x)", fontsize=8)
    if len(ref):
        row_axes[1].imshow(load_crop(ref.iloc[0].crop_path))
        row_axes[1].set_title(f"{pr.predicted} (correct example)", fontsize=8)
    for ax in row_axes:
        ax.axis("off")
fig.tight_layout()
fig.savefig(fig_dir / "confusion_pairs.png", dpi=150)
plt.show()

pairs.head(15)[["target", "target_name", "predicted", "predicted_name", "count"]]

## 5. 대표 오분류

확신도(`classifier_score`)가 높은데 틀린 샘플부터 보여 준다. 라벨 오류인지, 정말 비슷한 알약인지, crop 문제(잘림·옆 알약)인지 확인한다.

In [ ]:
n_show = 24
shown = errors.sort_values("classifier_score", ascending=False).head(n_show)
cols = 6
rows = int(np.ceil(len(shown) / cols)) or 1
fig, axes = plt.subplots(rows, cols, figsize=(2.3 * cols, 2.6 * rows), squeeze=False)
for ax in axes.flat:
    ax.axis("off")
for ax, r in zip(axes.flat, shown.itertuples()):
    ax.imshow(load_crop(r.crop_path))
    dup = " *" if r.near_dup_in_train else ""
    ax.set_title(f"true {classes.code[r.target_class_index]}\npred {classes.code[r.predicted_class_index]} {r.classifier_score:.2f}{dup}", fontsize=7)
fig.suptitle("High-confidence errors (* = near_dup_in_train)", fontsize=10)
fig.tight_layout()
fig.savefig(fig_dir / "top_errors.png", dpi=150)
plt.show()

shown[["sample_id", "target_class_index", "predicted_class_index", "classifier_score", "near_dup_in_train"]].assign(
    target=lambda d: d.target_class_index.map(classes["name"]),
    predicted=lambda d: d.predicted_class_index.map(classes["name"]),
).drop(columns=["target_class_index", "predicted_class_index"]).round(3)

In [ ]:
print("저장한 파일:")
for p in sorted(fig_dir.iterdir()):
    print(" ", p.name)